# Лабораторная работа 1 — рабочий ноутбук

**ФИО:** Мищенко Мария Михайловна

**Файлы данных:** S17: `clients.csv`, `transactions.csv`

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = "data/clients.csv"
RAW_TXN = "data/transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [5]:
# проверка кодировок на нескольких строках данных
print("----- Clients table -----")
with open(RAW_CLIENTS, encoding='utf-8') as f:
    for _ in range(3):
        print(repr(f.readline()))

print("----- Transactions table -----")
with open(RAW_TXN, encoding='cp1251') as f:
    for _ in range(3):
        print(repr(f.readline()))

----- Clients table -----
'client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in\n'
'C004755,1972-02-03,Ж,МОСКВА,Премиальный,Серебро,Кредитная,19.11.2020,03.04.2022,Не указан,,Да,2024-08,,00805,Да\n'
'C004627,1976-05-16,М,Новосибирск,Зарплатный,Серебро,Дебетовая,16.06.2012,31.08.2017,Зарплата в нашем банке,61600.0,Да,2025-05,,00256,Да\n'
----- Transactions table -----
'txn_id;client_id;txn_datetime;amount;currency;category;mcc_code;channel;status\n'
'990060;C004365;2023-12-05 14:19:43;145,82;RUB;Продукты;5411;Карта;успешно\n'
'990167;C005370;2023-12-10 10:57:55;1480,83;RUB;Продукты;5411;СБП;успешно\n'


**Вывод:** файл с клиентами читается как UTF-8, разделитель — ",", дробная часть — через точку, первая строка — заголовок. Параметры `read_csv` по умолчанию подходят. Файл с транзакциями имеет другую кодировку, разделитель - ";", дробная часть через запятую => явно указываем параметры, когда читаем этот файл.

In [9]:
clients_raw = pd.read_csv(RAW_CLIENTS)
clients = clients_raw.copy()
clients_raw.head()

,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C004755,1972-02-03,Ж,МОСКВА,Премиальный,Серебро,Кредитная,19.11.2020,03.04.2022,Не указан,NaN,Да,2024-08,NaN,805,Да
1,C004627,1976-05-16,М,Новосибирск,Зарплатный,Серебро,Дебетовая,16.06.2012,31.08.2017,Зарплата в нашем банке,61600.0,Да,2025-05,NaN,256,Да
2,C001846,1962-06-17,М,Москва,Зарплатный,Серебро,Дебетовая,10.07.2019,03.11.2021,Зарплата в нашем банке,102900.0,Нет,NaN,NaN,198,Да
3,C004355,1978-03-05,М,Москва,Зарплатный,Золото,Дебетовая,13.10.2024,08.02.2025,Зарплата в нашем банке,85100.0,Да,2024-04,NaN,927,Да
4,C004093,1992-10-13,Ж,СПб,Массовый,Базовый,Дебетовая,27.08.2015,24.09.2017,Зарплата в нашем банке,47900.0,Да,2024-03,NaN,171,Да


In [10]:
transactions_raw = pd.read_csv(RAW_TXN, encoding='cp1251', sep=';', decimal=',')
transactions = transactions_raw.copy()
transactions_raw.head()

,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990060,C004365,2023-12-05 14:19:43,145.82,RUB,Продукты,5411,Карта,успешно
1,990167,C005370,2023-12-10 10:57:55,1480.83,RUB,Продукты,5411,СБП,успешно
2,990142,C005487,2023-12-12,530.24,RUB,Продукты,5411,Карта,успешно
3,100001,C000952,2024-01-01 00:08:10,518.25,RUB,Кафе/рестораны,5812,Онлайн-банк,успешно
4,100002,C000498,2024-01-01 01:53:31,896.19,RUB,Продукты,5411,Онлайн-банк,успешно


## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

**Clients table**

In [26]:
print("Объём: ", clients.shape)
clients.head()

Объём:  (5715, 16)


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C004755,1972-02-03,Ж,МОСКВА,Премиальный,Серебро,Кредитная,19.11.2020,03.04.2022,Не указан,NaN,Да,2024-08,NaN,805,Да
1,C004627,1976-05-16,М,Новосибирск,Зарплатный,Серебро,Дебетовая,16.06.2012,31.08.2017,Зарплата в нашем банке,61600.0,Да,2025-05,NaN,256,Да
2,C001846,1962-06-17,М,Москва,Зарплатный,Серебро,Дебетовая,10.07.2019,03.11.2021,Зарплата в нашем банке,102900.0,Нет,NaN,NaN,198,Да
3,C004355,1978-03-05,М,Москва,Зарплатный,Золото,Дебетовая,13.10.2024,08.02.2025,Зарплата в нашем банке,85100.0,Да,2024-04,NaN,927,Да
4,C004093,1992-10-13,Ж,СПб,Массовый,Базовый,Дебетовая,27.08.2015,24.09.2017,Зарплата в нашем банке,47900.0,Да,2024-03,NaN,171,Да


In [96]:
clients_overview = pd.DataFrame({
    "dtype": clients.dtypes,
    "n_unique": clients.nunique(),
    "n_missing": clients.isna().sum(),
    'missing_%': (clients.isna().mean() * 100).round(2),
})
clients_overview

,dtype,n_unique,n_missing,missing_%
client_id,str,5660,0,0.00
birth_date,str,4846,0,0.00
gender,str,2,0,0.00
city,str,17,0,0.00
segment,str,4,0,0.00
loyalty_tier,str,4,0,0.00
card_type,str,3,0,0.00
registration_date,str,3267,0,0.00
card_open_date,str,2642,0,0.00
income_source,str,3,0,0.00


In [105]:
clients.describe(include=['float64'])

,monthly_income,satisfaction_score
count,5161.000000,2346.000000
mean,95110.636524,3.345269
std,83253.057907,1.388090
min,-999.000000,1.000000
25%,52600.000000,2.000000
50%,72800.000000,3.000000
75%,108000.000000,5.000000
max,991300.000000,5.000000


**Transactions table**

In [31]:
print("Объём: ", transactions.shape)
transactions.head()

Объём:  (616993, 9)


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990060,C004365,2023-12-05 14:19:43,145.82,RUB,Продукты,5411,Карта,успешно
1,990167,C005370,2023-12-10 10:57:55,1480.83,RUB,Продукты,5411,СБП,успешно
2,990142,C005487,2023-12-12,530.24,RUB,Продукты,5411,Карта,успешно
3,100001,C000952,2024-01-01 00:08:10,518.25,RUB,Кафе/рестораны,5812,Онлайн-банк,успешно
4,100002,C000498,2024-01-01 01:53:31,896.19,RUB,Продукты,5411,Онлайн-банк,успешно


In [32]:
transactions_overview = pd.DataFrame({
    "dtype": transactions.dtypes,
    "n_unique": transactions.nunique(),
    "n_missing": transactions.isna().sum()
})
transactions_overview

,dtype,n_unique,n_missing
txn_id,int64,615493,0
client_id,str,5991,0
txn_datetime,str,560854,0
amount,float64,324291,0
currency,str,3,0
category,str,12,0
mcc_code,int64,9,0
channel,str,4,0
status,str,3,0


In [99]:
transactions.describe(include=['float64'])

,amount
count,616993.000000
mean,1887.642972
std,2745.238445
min,-62286.590000
25%,468.710000
50%,1069.640000
75%,2275.320000
max,116052.630000


### 2.2. Дубли

In [44]:
print("Полные дубликаты строк в clients: ", clients.duplicated().sum())
print("Полные дубликаты строк в transactions: ", transactions.duplicated().sum())

Полные дубликаты строк в clients:  40
Полные дубликаты строк в transactions:  1283


### 2.3. Соответствие данных описанию полей из задания

**Clients**

In [76]:
clients.columns

Index(['client_id', 'birth_date', 'gender', 'city', 'segment', 'loyalty_tier', 'card_type', 'registration_date', 'card_open_date', 'income_source',
       'monthly_income', 'has_mobile_app', 'app_last_login', 'satisfaction_score', 'branch_code', 'marketing_opt_in'],
      dtype='str')

In [68]:
# 1. проверка соответсвия категорий в признаках, для которых указан чёткий список возможных значений категории
multi_category_columns = ["segment", "loyalty_tier", "card_type", "satisfaction_score"]
for col in multi_category_columns:
    print(f"{col}: {clients[col].unique().tolist()}\n")
    display(pd.DataFrame({
        'count': clients[col].value_counts(dropna=False),
        'freq': clients[col].value_counts(normalize=True).round(3)
    }))

# 2. проверка значений булевых признаков
boolean_columns = ["gender", "has_mobile_app", "marketing_opt_in"]
for bool_col in boolean_columns:
    print(f"\n{bool_col}: {clients[bool_col].unique().tolist()}\n")
    display(pd.DataFrame({
            'count': clients[bool_col].value_counts(dropna=False),
            'freq': clients[bool_col].value_counts(normalize=True).round(3)
    }))

segment: ['Премиальный', 'Зарплатный', 'Массовый', 'Private']



,count,freq
segment,,
Массовый,2956,0.517
Зарплатный,1604,0.281
Премиальный,921,0.161
Private,234,0.041


loyalty_tier: ['Серебро', 'Золото', 'Базовый', 'Платина']



,count,freq
loyalty_tier,,
Базовый,2856,0.500
Серебро,1605,0.281
Золото,936,0.164
Платина,318,0.056


card_type: ['Кредитная', 'Дебетовая', 'Премиальная']



,count,freq
card_type,,
Дебетовая,4036,0.706
Кредитная,1192,0.209
Премиальная,487,0.085


... (4 outputs left out)


has_mobile_app: ['Да', 'Нет']



,count,freq
has_mobile_app,,
Да,3513,0.615
Нет,2202,0.385



marketing_opt_in: ['Да', 'Нет']



,count,freq
marketing_opt_in,,
Да,5030,0.88
Нет,685,0.12


In [74]:
# 3. анализ значений в другие не числовых столбцах 
print(clients['city'].value_counts())

city
Москва                1134
Екатеринбург           643
Санкт-Петербург        619
Новосибирск            570
Нижний Новгород        565
Казань                 528
МОСКВА                 195
г. Москва              191
 москва                185
казань                 157
г. Новосибирск         157
г.Екатеринбург         151
Н. Новгород            150
ЕКАТЕРИНБУРГ           141
Санкт Петербург        125
г. Санкт-Петербург     105
СПб                     99
Name: count, dtype: int64


**Transactions**

In [84]:
transactions.columns

Index(['txn_id', 'client_id', 'txn_datetime', 'amount', 'currency', 'category', 'mcc_code', 'channel', 'status'], dtype='str')

In [104]:
# 1. проверка соответсвия категорий в признаках, для которых указан чёткий список возможных значений категории
multi_category_columns = ["channel", "status"]
for col in multi_category_columns:
    print(f"{col}: {transactions[col].unique()}\n")
    display(pd.DataFrame({
        'count': transactions[col].value_counts(dropna=False),
        'freq': transactions[col].value_counts(normalize=True).round(3)
    }))

channel: <StringArray>
['Карта', 'СБП', 'Онлайн-банк', 'Токен']
Length: 4, dtype: str



,count,freq
channel,,
Карта,315225,0.511
СБП,109742,0.178
Токен,104081,0.169
Онлайн-банк,87945,0.143


status: <StringArray>
['успешно', 'отклонено', 'возврат']
Length: 3, dtype: str



,count,freq
status,,
успешно,597249,0.968
отклонено,12169,0.020
возврат,7575,0.012


In [103]:
# 2. проверка других нечисловых столбцов
print(transactions['currency'].value_counts())
print("Кол-во возможных значений кода торговой точки: ",len(transactions['mcc_code'].value_counts()))

print(transactions['category'].value_counts())
print("Кол-во возможных значений категорий: ",len(transactions['category'].value_counts()))

currency
RUB    614514
USD      1290
EUR      1189
Name: count, dtype: int64
Кол-во возможных значений кода торговой точки:  9
category
Продукты               162838
Одежда и обувь          69436
Транспорт               65160
АЗС                     56577
Кафе и рестораны        47774
Развлечения и досуг     46552
Аптеки                  44667
Онлайн-сервисы          43467
Связь и интернет        29193
Связь и Интернет        19565
кафе и рестораны        15965
Кафе/рестораны          15799
Name: count, dtype: int64
Кол-во возможных значений категорий:  12


### 2.4. Связь между таблицами

In [92]:
# 1. совпадение типов 
print("Совпадение типов у полей с id: ", clients["client_id"].dtype == transactions["client_id"].dtype)
print(transactions["client_id"].dtype)

# 2. клиенты, которые есть в clients, но отсутствуют в transactions
clients_without_transactions = clients[~clients["client_id"].isin(transactions["client_id"])]
print("Кол-во клиентов без транзакций: ", len(clients_without_transactions))
display(clients_without_transactions)

# 3. транзакции клиентов, который нет в clients
empty_transactions = transactions[~transactions["client_id"].isin(clients["client_id"])]
print("Кол-во транзакций от клиентов не из clients: ", len(empty_transactions))
display(empty_transactions)

Совпадение типов у полей с id:  True
str
Кол-во клиентов без транзакций:  9


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
487,C000178,1937-05-31,Ж,Екатеринбург,Массовый,Базовый,Дебетовая,31.01.2025,06.05.2025,Не указан,48500.0,Нет,NaN,3.0,420,Да
2235,C001665,1951-03-19,Ж,Нижний Новгород,Массовый,Серебро,Кредитная,13.09.2019,22.04.2025,Не указан,50100.0,Да,2024-11,1.0,731,Да
2460,C002859,1971-05-12,Ж,МОСКВА,Массовый,Базовый,Дебетовая,28.12.2023,14.04.2025,Не указан,47200.0,Нет,NaN,5.0,184,Да
2957,C000855,1968-03-07,Ж,Екатеринбург,Массовый,Базовый,Дебетовая,07.08.2019,08.05.2025,Не указан,59200.0,Да,2024-11,NaN,721,Да
3175,C004481,1971-11-07,М,Москва,Премиальный,Золото,Дебетовая,03.07.2020,11.05.2025,Зарплата в нашем банке,292200.0,Да,2024-09,NaN,113,Да
3191,C005667,1994-10-07,М,Новосибирск,Зарплатный,Серебро,Дебетовая,03.02.2025,25.04.2025,Зарплата в нашем банке,53500.0,Да,2024-12,NaN,508,Да
5247,C005283,1949-03-21,М,Екатеринбург,Массовый,Базовый,Дебетовая,06.02.2021,30.04.2025,Не указан,NaN,Да,2024-10,NaN,883,Нет
5277,C002181,1975-10-30,Ж,г. Москва,Массовый,Базовый,Дебетовая,25.08.2023,13.05.2025,Зарплата в нашем банке,41400.0,Нет,NaN,NaN,538,Да
5554,C003462,1936-08-20,Ж,москва,Зарплатный,Серебро,Дебетовая,17.07.2024,29.04.2025,Зарплата в нашем банке,129600.0,Нет,NaN,NaN,652,Да


Кол-во транзакций от клиентов не из clients:  30422


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
4,100002,C000498,2024-01-01 01:53:31,896.19,RUB,Продукты,5411,Онлайн-банк,успешно
16,100013,C003574,2024-01-01 05:18:07,4912.64,RUB,Аптеки,5912,Карта,успешно
24,100021,C004476,2024-01-01,1211.23,RUB,Продукты,5411,Карта,успешно
46,100043,C002888,2024-01-01 07:48:53,805.26,RUB,Кафе и рестораны,5812,Онлайн-банк,успешно
51,100048,C005228,2024-01-01 07:55:00,906.36,RUB,Кафе и рестораны,5812,СБП,успешно
...,...,...,...,...,...,...,...,...,...
573132,990032,C004721,2025-05-16,1354.11,RUB,АЗС,5541,Онлайн-банк,успешно
616918,990126,C003390,2025-08-11 17:58:31,744.06,RUB,Продукты,5411,Карта,успешно
616937,990156,C001425,2025-10-18 16:48:07,100.62,RUB,Транспорт,4111,Карта,успешно
616959,990189,C003390,2026-01-19,407.60,RUB,Связь и Интернет,4814,Токен,успешно


**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

**Выводы:**

I. Таблица `clients`
1. Кол-во строк: 5715
2. Три признака представлены числовым типом данных, остальные имеют тип str. Столбцы, по смыслу являющиеся категориями, также хранятся как строки.
3. Пропущено больше половины значений в `satisfaction_score`(~59%), также пропуски в `monthly_income`(~10%), `app_last_login`(~39%).
4. Минимальное значение `monthly_income` отрицательное - странное значение для этого столбца; среднее значение больше медианы, а максимальное больше медианы в ~13 раз.
5. 40 полных дубликатов строк.
6. Признаки с явно указанными в условии категориями не имеют порпусков, а их категории соответствуют указанным в задании, но хранятся они как строки.
7. Бинарные по смыслу признаки хранятся как строки.
8. В `city` один и тот же город записан разными способами, что значительно увеличивает кол-во уникальных значений этого столбца.


II. Таблица `transactions`
1. Кол-во строк: 616993
2. `txn_id`, `amount`, `mcc_code` хранятся числовым типом данных, остальные - строки.
3. В `amount`: среднее и медиана отличаются не сильно; минимальное значение отрицательное - подозрительное значение; максимальное значение ~116k при медиане ~1.8k => правосторонняя асимметрия и, возможно, выбросы.
4. Пропусков в данных нет
5. 1283 полных дубликатов строк
6. Признаки с явно указанными в условии категориями (`channel`, `status`) не имеют порпусков, а их категории соответствуют указанным в задании. По смыслу являются категориями, но хранятся они как строки. 
7. В столбце `category` одни и те же категории записаны различными способами, что некорректно увеличивает число уникальных категорий: становится 12 вместо указанных 9. Хранится типом str.


III.  Связь между таблицами
- Идентификатор, связывающий таблицы - client_id, имеет тип данных str в обеих таблицах.
- Таблицы `clients` и `transactions` имею разное кол-во уникальных клиентов(client_id). В `clients` их 5660, а в `transactions` - 5991. В таблице `clients` присутствуют 9 клиентов, не совершавших вообще никаких транзакций, а в таблице `transactions` - 30422 транзакций соответствуют клиентам, которых нет в таблице clients. Это важно будет учитывать в случае соединения таблиц, чтобы корректно его выполнить.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

In [ ]:
#  даты, которые у нас в формате str -> datetime/date
# boolean из str -> true/false

### 3.2. Категориальные признаки

In [ ]:
# посмотреть, какие из них булевы и заменить строки на 0/1
# преобразовать city через словарь

### 3.3. Дубликаты

In [ ]:
# Полные дубли строк в данном случае не является реальным повторением события, 
# а скорее просто сбой/ошибка, поэтому такие дубли удаляем. 

### 3.4. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

### 3.5. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

In [ ]:
# даты регистрации и транзакций - сопоставить и сравнить

### 3.6. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [ ]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

In [ ]:
# посмотреть соотношения категорий

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [ ]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")